# Selective prediction under post-training quantisation

Abdelkerim Ali Hassan, GH1046256. MSc Data Science, Artificial Intelligence and
Digital Business, Gisma University of Applied Sciences.

This notebook reproduces every result and figure in the dissertation from
committed data. Two corpus variants, five seeds, three numeric precisions.
Runtime is about two minutes.

In [ ]:
import os, pathlib, subprocess, sys

if not pathlib.Path("quantised-calibration").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Alisafi66/quantised-calibration.git"], check=True)
os.chdir("quantised-calibration")
sys.path.insert(0, "src")

import importlib.util, json
import numpy as np
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score

from qcal.metrics import (expected_calibration_error, per_class_ece, aurc,
                          coverage_and_precision, threshold_for_target_precision)
from qcal.data import AG_NEWS_CLASSES

_spec = importlib.util.spec_from_file_location("figures", "scripts/make_figures.py")
figures = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(figures)

SEEDS, VARIANTS = [0, 1, 2, 3, 4], ["balanced", "long_tailed"]
PRECISIONS, PROBS = ["fp32", "int8", "int4"], pathlib.Path("results/probs")
T_CRIT = 2.776   # two-sided, 5 per cent, 4 degrees of freedom

def load(variant, seed):
    return (np.load(PROBS / f"{variant}_seed{seed}_test.npz"),
            np.load(PROBS / f"{variant}_seed{seed}_prob_calib.npz"))

def conf_corr(npz, precision):
    pr = npz[precision]
    return pr.max(1), (pr.argmax(1) == npz["label"]).astype(float)

def paired_t(d):
    d = np.asarray(d, float)
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    return d.mean(), t, bool(np.all(d > 0) or np.all(d < 0))

def verdict(t):
    return "significant" if abs(t) > T_CRIT else "not significant"

print("ready,", len(list(PROBS.glob("*.npz"))), "probability files")

## The data

AG News, pinned by SHA-256. The training split is used in two forms: balanced,
and a long tail built by keeping 100, 30, 10 and 3 per cent of each class.
The test split stays balanced in both cases, so per-class quantities have
enough examples to measure.

In [ ]:
manifest = json.load(open("data/processed/manifest.json"))
print(manifest["corpus"], "pinned", manifest["pinned_utc"], "\n")
for name, m in manifest["splits"].items():
    print(f'{name:20s} n={m["n"]:>7,}  {m["sha256"][:16]}  {m["class_counts"]}')

ex = np.load(PROBS / "long_tailed_seed0_test.npz")
print(f"\nEach file holds {list(ex.keys())}, probabilities of shape {ex['fp32'].shape}.")
print("Same inputs in the same order at three precisions, so every comparison")
print("below is paired against the FP32 model trained from the same seed.")

## RQ1. Does quantisation distort confidence estimates?

> *How far does post-training quantisation distort a model's confidence
> estimates, independently of any loss in accuracy?*

Expected calibration error, 15 equal-width bins. Macro-F1 is recorded beside
it as an observed covariate, not as a filter.

In [ ]:
print(f"{'variant':13s} {'precision':9s} {'macro-F1':>9s} {'ECE':>8s} {'AURC':>8s} {'mean conf':>10s}")
print("-" * 62)
for variant in VARIANTS:
    for p in PRECISIONS:
        f1s, eces, aurcs, confs = [], [], [], []
        for s in SEEDS:
            test, _ = load(variant, s)
            conf, corr = conf_corr(test, p)
            f1s.append(f1_score(test["label"], test[p].argmax(1), average="macro"))
            eces.append(expected_calibration_error(conf, corr))
            aurcs.append(aurc(conf, corr)); confs.append(conf.mean())
        print(f"{variant:13s} {p:9s} {np.mean(f1s):9.4f} {np.mean(eces):8.4f} "
              f"{np.mean(aurcs):8.4f} {np.mean(confs):10.4f}")
    print()

In [ ]:
print("Paired within-seed change from FP32\n")
for variant in VARIANTS:
    print(variant)
    for p in ("int8", "int4"):
        d_ece, d_f1 = [], []
        for s in SEEDS:
            test, _ = load(variant, s)
            d_ece.append(expected_calibration_error(*conf_corr(test, p))
                         - expected_calibration_error(*conf_corr(test, "fp32")))
            d_f1.append(f1_score(test["label"], test[p].argmax(1), average="macro")
                        - f1_score(test["label"], test["fp32"].argmax(1), average="macro"))
        m, t, c = paired_t(d_ece); mf, tf, cf = paired_t(d_f1)
        print(f"  {p}  dECE {m:+.5f}  t(4)={t:+6.2f}  {verdict(t):15s} same sign: {c}")
        print(f"      dF1  {mf:+.5f}  t(4)={tf:+6.2f}  {verdict(tf):15s} same sign: {cf}")
    print()

In [ ]:
figures.fig_what_ece_misses();

**No detectable distortion.** Every ECE t-statistic falls below the critical
value and the signs reverse between seeds, on both variants and at both
precisions.

INT8 does lose accuracy consistently. The calibration measure does not register
it, because INT8 loses accuracy and loses confidence at the same time and the
two cancel in an averaged figure. The right-hand panel shows what a different
measure sees in the same runs.

### RQ1(b). Is the distortion concentrated in low-frequency classes?

In [ ]:
sizes = {"balanced": [28500]*4, "long_tailed": [28500, 8550, 2850, 855]}
for variant in VARIANTS:
    print(variant)
    for p in PRECISIONS:
        per_seed = []
        for s in SEEDS:
            test, _ = load(variant, s)
            pc = per_class_ece(*conf_corr(test, p), test["label"], 4)
            per_seed.append([pc[c] for c in range(4)])
        mean = np.mean(per_seed, axis=0)
        print(f"  {p:5s} " + "  ".join(
              f"{AG_NEWS_CLASSES[c]}[{sizes[variant][c]:>5}] {mean[c]:.4f}" for c in range(4)))
    base = np.mean([[per_class_ece(*conf_corr(load(variant, s)[0], "fp32"),
                                   load(variant, s)[0]["label"], 4)[c] for c in range(4)]
                    for s in SEEDS], axis=0)
    print(f"  rare to common ratio at FP32: {base[3]/base[0]:.1f}x\n")

In [ ]:
figures.fig_per_class();

**Yes, strongly, but it is an effect of imbalance rather than of quantisation.**
The rarest class carries roughly sixteen times the calibration error of the
most common at full precision, before any compression. Quantisation adds to the
rare class but does not create the gap.

## RQ2. Does an abstention threshold survive quantisation?

> *How far does an abstention threshold tuned at full precision fail to deliver
> its promised precision after quantisation?*

A threshold is fitted on the full-precision model's held-out calibration split
to reach a 95 per cent precision target, then applied unchanged to each
quantised model. Two things are measured: the precision actually delivered, and
the fraction of inputs still answered.

In [ ]:
for variant in VARIANTS:
    print(f"{variant}\n" + "-" * 46)
    print(f"{'precision':10s} {'coverage':>9s} {'delivered':>10s} {'promise':>9s}")
    for p in PRECISIONS:
        cov, prec = [], []
        for s in SEEDS:
            test, cal = load(variant, s)
            th = threshold_for_target_precision(*conf_corr(cal, "fp32"), 0.95)
            c, pp = coverage_and_precision(*conf_corr(test, p), th)
            cov.append(c); prec.append(pp)
        c, pp = np.mean(cov), np.mean(prec)
        print(f"{p:10s} {c:9.4f} {pp:10.4f} {'kept' if pp >= 0.95 else 'missed':>9s}")
    print()

In [ ]:
for target in (0.95, 0.98):
    print(f"coverage change at a {target:.0%} precision target")
    for variant in VARIANTS:
        for p in ("int8", "int4"):
            d = []
            for s in SEEDS:
                test, cal = load(variant, s)
                th = threshold_for_target_precision(*conf_corr(cal, "fp32"), target)
                base = coverage_and_precision(*conf_corr(test, "fp32"), th)[0]
                d.append(100 * (coverage_and_precision(*conf_corr(test, p), th)[0] - base))
            m, t, c = paired_t(d)
            print(f"  {variant:13s} {p}  {m:+7.2f} pp  t(4)={t:+7.2f}  "
                  f"{verdict(t):15s} same sign: {c}")
    print()

In [ ]:
figures.fig_coverage_by_target();

**The threshold keeps its precision promise and pays in coverage.** The
promised-minus-delivered shortfall is a fraction of a percentage point, barely
larger than the full-precision model's own generalisation gap.

What collapses is the fraction of inputs answered: around eight percentage
points at a 95 per cent target on the long-tailed split and eleven at 98 per
cent, with the same sign in every seed. INT4 does not.

This is the opposite failure mode from the one the question anticipated. The
model does not become reckless. It becomes uselessly cautious.

## Why no threshold can repair it

Three checks separate a misplaced operating point from a genuine loss of
capability: re-tune the threshold on the quantised model's own calibration
split, fit an oracle threshold on the test set itself as an upper bound on what
any threshold could achieve, and measure AURC, which is threshold-free and
reflects how well a model orders its own correctness.

In [ ]:
for variant in VARIANTS:
    print(f"{variant}\n" + "-" * 68)
    for p in ("int8", "int4"):
        naive, retuned, oracle, d_aurc = [], [], [], []
        for s in SEEDS:
            test, cal = load(variant, s)
            th_fp = threshold_for_target_precision(*conf_corr(cal, "fp32"), 0.95)
            base = coverage_and_precision(*conf_corr(test, "fp32"), th_fp)[0]
            qc, qk = conf_corr(test, p)
            naive.append(100 * (coverage_and_precision(qc, qk, th_fp)[0] - base))
            th_q = threshold_for_target_precision(*conf_corr(cal, p), 0.95)
            retuned.append(100 * (coverage_and_precision(qc, qk, th_q)[0] - base))
            b_or = coverage_and_precision(*conf_corr(test, "fp32"),
                      threshold_for_target_precision(*conf_corr(test, "fp32"), 0.95))[0]
            oracle.append(100 * (coverage_and_precision(qc, qk,
                      threshold_for_target_precision(qc, qk, 0.95))[0] - b_or))
            d_aurc.append(aurc(qc, qk) - aurc(*conf_corr(test, "fp32")))
        for label, vals in (("carried unchanged", naive), ("re-tuned", retuned),
                            ("oracle threshold", oracle)):
            m, t, _ = paired_t(vals)
            print(f"  {p}  {label:20s} coverage {m:+7.2f} pp   t(4)={t:+7.2f}")
        m, t, _ = paired_t(d_aurc)
        print(f"  {p}  {'AURC':20s}          {m:+.5f}      t(4)={t:+7.2f}  {verdict(t)}\n")

In [ ]:
figures.fig_risk_coverage();

Re-tuning does not recover the coverage, and neither does an oracle threshold
fitted on the test set itself. **AURC degrades significantly**, so the model has
become worse at ordering its own correctness, and the curve above has moved
rather than the operating point on it.

Calibration and discrimination are orthogonal. Expected calibration error asks
whether a confidence of 0.9 is right ninety per cent of the time. AURC asks
whether more confident predictions are in fact more likely right. Selective
prediction depends on the second, and quantisation damaged the second while
leaving the first intact.

## RQ3(a). Does temperature scaling repair it?

Temperature scaling is the standard remedy for miscalibration: divide the
logits by one scalar fitted on held-out data. Because softmax is
shift-invariant, the logits are recoverable from the stored probabilities as
their logarithm.

A single temperature rescales every logit vector by the same amount. It can
shift the ranking slightly, since examples whose logits are spread differently
respond differently, but it cannot restructure it. If the damage is in the
ordering, ECE should improve while the coverage gap survives.

In [ ]:
def temp_scale(prob, T):
    z = np.log(np.clip(prob, 1e-12, None)) / T
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(prob, y):
    def nll(log_t):
        p = temp_scale(prob, np.exp(log_t))
        return -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None)).mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-2, 2), method="bounded").x))

variant = "long_tailed"
print(f"{variant}, temperature fitted on each model's own calibration split\n")
for p in ("int8", "int4"):
    Ts, d_ece, d_aurc, gap = [], [], [], []
    for s in SEEDS:
        test, cal = load(variant, s)
        y, yc = test["label"], cal["label"]
        T = fit_temperature(cal[p], yc); Ts.append(T)
        qk = (test[p].argmax(1) == y).astype(float)
        st, sc = temp_scale(test[p], T), temp_scale(cal[p], T)
        d_ece.append(expected_calibration_error(st.max(1), qk)
                     - expected_calibration_error(test[p].max(1), qk))
        d_aurc.append(aurc(st.max(1), qk) - aurc(test[p].max(1), qk))
        base = coverage_and_precision(*conf_corr(test, "fp32"),
                 threshold_for_target_precision(*conf_corr(cal, "fp32"), 0.95))[0]
        th = threshold_for_target_precision(sc.max(1), (sc.argmax(1) == yc).astype(float), 0.95)
        gap.append(100 * (coverage_and_precision(st.max(1), qk, th)[0] - base))
    print(f"  {p}  T = {np.mean(Ts):.3f}")
    for label, vals, fmt in (("ECE change from scaling", d_ece, "+.5f"),
                             ("AURC change from scaling", d_aurc, "+.6f"),
                             ("coverage vs FP32 after scaling (pp)", gap, "+.2f")):
        m, t, _ = paired_t(vals)
        print(f"      {label:38s} {m:{fmt}}  t(4)={t:+6.2f}  {verdict(t)}")
    print()

In [ ]:
figures.fig_temperature();

**Temperature scaling repairs the calibration and does not repair the failure.**
ECE falls by roughly half. AURC shifts by less than 0.001 against the +0.010
quantisation introduced, so the ranking is essentially untouched, and the
coverage gap is unchanged.

The remedy the literature prescribes for miscalibration cannot help here,
because miscalibration is not what occurred.

## Summary

| Question | Answer |
|---|---|
| Does quantisation distort confidence estimates? | Not detectably, by expected calibration error |
| Is the distortion concentrated in rare classes? | Yes, roughly sixteenfold, but present before quantisation |
| Does a full-precision threshold survive? | It keeps its precision and loses up to 11 points of coverage |
| Can re-tuning restore it? | No, and neither can an oracle threshold |
| Does temperature scaling help? | It halves ECE and leaves the failure untouched |

Post-training quantisation degrades selective-prediction behaviour in a way
that expected calibration error cannot detect, and the degradation tracks the
quantisation scheme rather than the bit width. Static INT8 damaged the model's
ability to rank its own correctness; weight-only block-wise INT4 did not.